In [ ]:
!pip install simpletransformers
!pip install transformers-interpret

import pandas
import numpy
from sklearn.metrics import f1_score, classification_report
from simpletransformers.classification import ClassificationModel
from transformers_interpret import SequenceClassificationExplainer

In [ ]:
trainPath = "../input/normalizedgoemotions/GoE-Nov17/newTrainData.csv"
testPath = "../input/normalizedgoemotions/GoE-Nov17/newTestData.csv"

emoList = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 
'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

In [ ]:
train_df = pandas.read_csv(trainPath)
train_df = train_df.sample(frac=1).reset_index(drop=True)

newCol = []
for label in train_df['label']:
    newCol.append(emoList.index(label))
train_df['label_desc'] = train_df['label']
train_df['label'] = newCol
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

train_df.head()

In [ ]:
test_df = pandas.read_csv(testPath)
test_df = test_df.sample(frac=1).reset_index(drop=True)

newCol = []
for label in test_df['label']:
    newCol.append(emoList.index(label))
test_df['label_desc'] = test_df['label']
test_df['label'] = newCol
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

test_df.head()

In [ ]:
model = ClassificationModel('roberta', 'roberta-base', num_labels=len(emoList), 
                                      args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,                                         
                                            'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                            'num_train_epochs': 1, "reprocess_input_data": True, 'max_seq_length': 64})

model.train_model(train_df)

In [ ]:
predictions, raw_outputs = model.predict(list(test_df['text']))
print(f"F1-Micro Score - {f1_score(list(test_df['labels']), predictions, average='micro')}")
print(f"F1-Macro Score - {f1_score(list(test_df['labels']), predictions, average='macro')}")
classification_report(test_df['labels'], predictions, target_names=emoList, digits=len(emoList))

In [ ]:
idx = 16
cls_explainer = SequenceClassificationExplainer(model.model, model.tokenizer, custom_labels=emoList)
word_attributions = cls_explainer(test_df.iloc[idx]['text'])
label = emoList[int(cls_explainer.predicted_class_index)]

print(f"Text : {test_df.iloc[idx]['text']}")
print('-'*100)
print(f"True Label : {test_df.iloc[idx]['label_desc']}")
print(f"Predicted Label : {label}")
print('-'*100)

cls_explainer.visualize()